#### Bronze ingest (v2, with engineered ML signal)

Same as 01_bronze_ingest, but the generator now produces data with DOCUMENTED
CAUSAL RELATIONSHIPS (cost overrun, delay, safety) instead of pure noise, so
the ML models have real structure to recover. The data-quality mess is
unchanged, so the silver cleanup story still holds.

#### Cell 1

In [1]:
# Install Faker
%pip install faker

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 8, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



#### Cell 2

In [2]:
# Imports, seeds, output path
import os, random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
N_PROJECTS = 120

random.seed(SEED); np.random.seed(SEED); Faker.seed(SEED)
fake = Faker()

OUTDIR = "/lakehouse/default/Files/bronze/construction_raw"   # pandas writes via local mount
os.makedirs(OUTDIR, exist_ok=True)
print(f"Output: {OUTDIR}")

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 10, Finished, Available, Finished, False)

Output: /lakehouse/default/Files/bronze/construction_raw


#### Cell 3

In [3]:
# Reference data + engineered-signal coefficients
PROJECT_TYPES = ["Healthcare","Commercial","Higher Education","K-12 Education",
    "Government/Civic","Aviation","Data Center","Mission Critical","Industrial","Sports & Entertainment"]
DELIVERY_METHODS = ["CM at Risk","Design-Build","Design-Bid-Build","IPD","CM Agency"]
REGIONS = ["Kansas City","Denver","Dallas","Houston","Atlanta","Phoenix",
    "Portland","Nashville","Minneapolis","Omaha","Austin","Charlotte"]
CSI_DIVISIONS = {"01":"General Requirements","02":"Existing Conditions","03":"Concrete",
    "04":"Masonry","05":"Metals","06":"Wood, Plastics & Composites","07":"Thermal & Moisture Protection",
    "08":"Openings","09":"Finishes","21":"Fire Suppression","22":"Plumbing","23":"HVAC",
    "26":"Electrical","27":"Communications","31":"Earthwork","32":"Exterior Improvements","33":"Utilities"}
TRADES = ["Carpenter","Electrician","Plumber","Ironworker","Laborer","Operator",
    "Concrete Finisher","HVAC Tech","Foreman","Superintendent"]
INCIDENT_TYPES = ["Slip/Trip/Fall","Struck-by","Caught-between","Electrical","Fall from Height",
    "Laceration","Strain/Sprain","Near Miss","Equipment Damage","Heat Illness"]
SEVERITY = ["First Aid","Recordable","Lost Time","Near Miss"]

# --- engineered signal coefficients (the ground-truth relationships) ---
DELIVERY_OVERRUN = {"Design-Bid-Build":0.18,"CM Agency":0.10,"CM at Risk":0.05,"Design-Build":0.02,"IPD":-0.02}
TYPE_COMPLEXITY = {"Data Center":0.15,"Mission Critical":0.14,"Healthcare":0.12,"Aviation":0.10,
    "Higher Education":0.06,"Sports & Entertainment":0.08,"Commercial":0.04,"Industrial":0.05,
    "Government/Civic":0.05,"K-12 Education":0.03}
DIVISION_OVERRUN = {"23":0.12,"26":0.10,"22":0.09,"27":0.07,"21":0.05,"05":0.04,"03":0.03}
WINTER_MONTHS = {12,1,2}
TRADE_SEVERITY = {"Ironworker":0.30,"Operator":0.25,"Electrician":0.15,"HVAC Tech":0.12}


StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 11, Finished, Available, Finished, False)

#### Cell 4

In [4]:
# Mess helpers
def maybe_missing(v, p=0.05): return v if random.random() > p else None

def messy_date(dt, p_format=0.3, p_impossible=0.02):
    if dt is None: return None
    if random.random() < p_impossible:
        return random.choice(["2099-13-01","0001-01-01","13/45/2021",""])
    fmts = ["%Y-%m-%d","%m/%d/%Y","%d-%b-%Y","%m-%d-%y","%Y/%m/%d","%b %d, %Y"]
    weights = [0.5,0.2,0.1,0.1,0.05,0.05] if random.random() < p_format else [1,0,0,0,0,0]
    return dt.strftime(random.choices(fmts, weights=weights)[0])
    
def dirty_name(name, p=0.25):
    if random.random() > p: return name
    return random.choice([name.upper(),name.lower(),f"  {name} ",name.replace(",",""),
        name.replace("LLC","L.L.C."),name.replace("Inc","Inc."),name+" ",name.replace(" ","  ")])


StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 12, Finished, Available, Finished, False)

#### Cell 5

In [5]:
# Generators (outcomes carry engineered signal)
def gen_projects(n):
    rows, drivers = [], {}
    for i in range(1, n+1):
        pid = f"PRJ-{i:05d}"
        ptype = random.choice(PROJECT_TYPES)
        delivery = random.choice(DELIVERY_METHODS)
        base = np.random.lognormal(16.3, 0.9)
        if ptype in ("Healthcare","Data Center","Mission Critical"): base *= random.uniform(1.5,3.0)
        contract_value = round(base, 2)
        start = fake.date_between(start_date="-5y", end_date="-6M")
        planned_dur = random.randint(180,1100)
        planned_end = start + timedelta(days=planned_dur)
        # engineered delay: winter + complexity + delivery
        delay_factor = TYPE_COMPLEXITY.get(ptype,0.05)
        if start.month in WINTER_MONTHS: delay_factor += 0.10
        if delivery == "Design-Bid-Build": delay_factor += 0.08
        slip_pct = delay_factor + np.random.normal(0,0.06)
        actual_dur = max(int(planned_dur*(1+slip_pct)), int(planned_dur*0.9))
        actual_end = start + timedelta(days=actual_dur)
        status = random.choices(["Active","Complete","On Hold","Closed"], weights=[0.35,0.5,0.05,0.1])[0]
        sub_quality = round(random.uniform(2.0,5.0), 2)
        drivers[pid] = {"ptype":ptype,"delivery":delivery,"start_month":start.month,
            "planned_dur":planned_dur,"actual_dur":actual_dur,"sub_quality":sub_quality,
            "contract_value":contract_value,"start_date_obj":start}
        cv_out = contract_value
        if random.random() < 0.02: cv_out = random.choice([-contract_value,0,contract_value*50])
        rows.append({"project_id":pid,
            "project_name":maybe_missing(f"{fake.company()} {ptype} Facility",0.02),
            "project_type":maybe_missing(ptype,0.03),"region":random.choice(REGIONS),
            "delivery_method":delivery,"contract_value":cv_out,
            "start_date":messy_date(start),"planned_end_date":messy_date(planned_end),
            "actual_end_date":messy_date(actual_end) if status in ("Complete","Closed") else None,
            "square_footage":maybe_missing(random.randint(15000,900000),0.04),
            "status":status,"project_manager":maybe_missing(fake.name(),0.03)})
    df = pd.DataFrame(rows)
    df = pd.concat([df, df.sample(frac=0.02, random_state=1)], ignore_index=True)
    return df, drivers

def gen_cost_line_items(drivers):
    rows, lid = [], 1
    for pid, d in drivers.items():
        cv = d["contract_value"]
        if cv <= 0: cv = random.uniform(5e6,3e7)
        divs = random.sample(list(CSI_DIVISIONS.keys()), k=random.randint(6,len(CSI_DIVISIONS)))
        w = np.random.dirichlet(np.ones(len(divs)))
        for code, frac in zip(divs, w):
            budget = round(cv*frac, 2)
            change_orders = round(budget*random.uniform(0,0.20),2) if random.random()<0.4 else 0.0
            co_ratio = change_orders/budget if budget>0 else 0
            overrun = 1.0
            overrun += DELIVERY_OVERRUN.get(d["delivery"],0.05)
            overrun += TYPE_COMPLEXITY.get(d["ptype"],0.05)
            overrun += DIVISION_OVERRUN.get(code,0.0)
            overrun += 0.5*co_ratio
            overrun -= 0.04*(d["sub_quality"]-3.5)
            overrun += np.random.normal(0,0.07)
            overrun = max(overrun,0.80)
            actual = round(budget*overrun, 2)
            if random.random()<0.03: actual = random.choice([None,-actual,0])
            rows.append({"line_item_id":f"CLI-{lid:07d}","project_id":pid,
                "csi_division":code,"division_name":CSI_DIVISIONS[code],
                "budget_amount":maybe_missing(budget,0.02),"actual_amount":actual,
                "change_order_amount":change_orders,
                "cost_code_notes":maybe_missing(fake.sentence(nb_words=4),0.7)})
            lid += 1
    return pd.DataFrame(rows)

def gen_schedule_tasks(drivers):
    rows, tid = [], 1
    tnames = ["Mobilization","Site Prep","Foundations","Structural Steel","Concrete Pour","Roofing",
        "MEP Rough-in","Drywall","Finishes","Commissioning","Punch List","Substantial Completion"]
    for pid, d in drivers.items():
        cursor, prev = d["start_date_obj"], None
        proj_slip = d["actual_dur"]/d["planned_dur"]
        for tn in tnames:
            planned = random.randint(10,90)
            actual = max(1, int(planned*(proj_slip+np.random.normal(0,0.05))))
            rows.append({"task_id":f"TSK-{tid:07d}","project_id":pid,"task_name":tn,
                "predecessor_task":prev,"planned_start":messy_date(cursor,p_impossible=0.0),
                "planned_duration_days":planned,"actual_duration_days":maybe_missing(actual,0.05),
                "percent_complete":maybe_missing(random.choice([0,25,50,75,100,random.randint(0,100)]),0.03)})
            prev, cursor = tn, cursor+timedelta(days=actual); tid += 1
    return pd.DataFrame(rows)

def gen_labor_timesheets(drivers):
    rows, wid, ot_by_project = [], 1, {}
    for pid, d in drivers.items():
        ot_bias = TYPE_COMPLEXITY.get(d["ptype"],0.05); proj_ot = 0
        for _ in range(random.randint(15,40)):
            reg = random.choice([8,8,8,10,4,12])
            ot = random.choices([0,2,4,6], weights=[1-ot_bias,ot_bias,ot_bias*0.7,ot_bias*0.4])[0]
            proj_ot += ot
            rows.append({"timesheet_id":f"TS-{wid:08d}","project_id":pid,
                "worker_name":maybe_missing(fake.name(),0.04),"trade":random.choice(TRADES),
                "work_date":messy_date(fake.date_between(start_date="-3y",end_date="today")),
                "regular_hours":reg,"overtime_hours":ot,
                "hourly_rate":maybe_missing(round(random.uniform(28,78),2),0.03)}); wid += 1
        ot_by_project[pid] = proj_ot
    return pd.DataFrame(rows), ot_by_project

def gen_subcontractors(n=80):
    rows, names = [], []
    for i in range(1, n+1):
        sid = f"SUB-{i:04d}"
        raw = f"{fake.company()} {random.choice(['LLC','Inc','Contractors','Construction','Mechanical','Electric Co'])}"
        names.append((sid, raw))
        rows.append({"sub_id":sid,"sub_name":dirty_name(raw),
            "trade_focus":random.choice(list(CSI_DIVISIONS.values())),"region":random.choice(REGIONS),
            "performance_rating":maybe_missing(round(random.uniform(2.0,5.0),1),0.08),
            "prequalified":random.choice(["Y","N","Yes","No","TRUE","FALSE",None])})
    df = pd.DataFrame(rows)
    for sid, raw in random.sample(names, k=int(n*0.15)):
        df = pd.concat([df, pd.DataFrame([{"sub_id":f"{sid}-DUP","sub_name":dirty_name(raw,p=1.0),
            "trade_focus":random.choice(list(CSI_DIVISIONS.values())),"region":random.choice(REGIONS),
            "performance_rating":round(random.uniform(2.0,5.0),1),
            "prequalified":random.choice(["Y","N"])}])], ignore_index=True)
    return df

def gen_sub_bids(drivers, subs):
    rows, bid = [], 1; sids = subs["sub_id"].tolist()
    for pid in drivers:
        for code in random.sample(list(CSI_DIVISIONS.keys()), k=random.randint(3,8)):
            for _ in range(random.randint(1,5)):
                rows.append({"bid_id":f"BID-{bid:07d}","project_id":pid,"sub_id":random.choice(sids),
                    "csi_division":code,"bid_amount":round(np.random.lognormal(14.5,0.6),2),
                    "awarded":random.choice(["Y","N","N","N"]),
                    "bid_date":messy_date(fake.date_between(start_date="-4y",end_date="today"))}); bid += 1
    return pd.DataFrame(rows)

def gen_safety_incidents(drivers, ot_by_project):
    rows, iid = [], 1
    max_ot = max(ot_by_project.values()) if ot_by_project else 1
    for pid, d in drivers.items():
        ot_norm = ot_by_project.get(pid,0)/max_ot if max_ot else 0
        size_norm = min(d["contract_value"]/3e7, 2.0)
        rate = 0.6 + 2.0*ot_norm + 0.8*size_norm
        for _ in range(np.random.poisson(rate)):
            trade = random.choice(TRADES); sev_bias = TRADE_SEVERITY.get(trade,0.0)
            sev = random.choices(SEVERITY, weights=[0.5-sev_bias*0.5,0.25,0.10+sev_bias,0.15])[0]
            rows.append({"incident_id":f"INC-{iid:06d}","project_id":pid,
                "incident_date":messy_date(fake.date_between(start_date="-3y",end_date="today")),
                "incident_type":random.choice(INCIDENT_TYPES),"severity":sev,"trade_involved":trade,
                "lost_days":maybe_missing(random.randint(0,30) if sev=="Lost Time" else 0,0.05),
                "description":maybe_missing(fake.sentence(nb_words=8),0.3),
                "root_cause":maybe_missing(random.choice(["Housekeeping","PPE","Training","Equipment","Weather","Unknown"]),0.2)}); iid += 1
    return pd.DataFrame(rows)

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 13, Finished, Available, Finished, False)

#### Cell 6

In [6]:
# Generate all tables (note the driver-passing wiring)
projects, drivers = gen_projects(N_PROJECTS)
subs = gen_subcontractors()
costs = gen_cost_line_items(drivers)
schedule = gen_schedule_tasks(drivers)
labor, ot_by_project = gen_labor_timesheets(drivers)
bids = gen_sub_bids(drivers, subs)
safety = gen_safety_incidents(drivers, ot_by_project)

tables = {"projects":projects,"subcontractors":subs,"cost_line_items":costs,
    "schedule_tasks":schedule,"labor_timesheets":labor,"sub_bids":bids,"safety_incidents":safety}

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 14, Finished, Available, Finished, False)

#### Cell 7

In [7]:
# Write raw CSVs to bronze
for name, df in tables.items():
    path = f"{OUTDIR}/{name}.csv"
    df.to_csv(path, index=False)
    print(f"{name:20s} {len(df):>7,} rows -> {path}")
print(f"\nBronze v2 (signal-bearing) landing complete: {sum(len(d) for d in tables.values()):,} rows.")

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 15, Finished, Available, Finished, False)

projects                 122 rows -> /lakehouse/default/Files/bronze/construction_raw/projects.csv
subcontractors            92 rows -> /lakehouse/default/Files/bronze/construction_raw/subcontractors.csv
cost_line_items        1,379 rows -> /lakehouse/default/Files/bronze/construction_raw/cost_line_items.csv
schedule_tasks         1,440 rows -> /lakehouse/default/Files/bronze/construction_raw/schedule_tasks.csv
labor_timesheets       3,411 rows -> /lakehouse/default/Files/bronze/construction_raw/labor_timesheets.csv
sub_bids               1,917 rows -> /lakehouse/default/Files/bronze/construction_raw/sub_bids.csv
safety_incidents         189 rows -> /lakehouse/default/Files/bronze/construction_raw/safety_incidents.csv

Bronze v2 (signal-bearing) landing complete: 8,550 rows.


#### Cell 8

In [8]:
# Sanity peek: confirm the engineered signal is present in raw data
import pandas as pd
_p = pd.read_csv(f"{OUTDIR}/projects.csv").drop_duplicates("project_id")
_c = pd.read_csv(f"{OUTDIR}/cost_line_items.csv")
_c["budget_amount"] = pd.to_numeric(_c["budget_amount"], errors="coerce")
_c["actual_amount"] = pd.to_numeric(_c["actual_amount"], errors="coerce")
_m = _c[(_c.budget_amount>0)&(_c.actual_amount>0)].merge(_p[["project_id","delivery_method"]], on="project_id")
_m["overrun"] = _m.actual_amount/_m.budget_amount
print("Mean overrun by delivery method (engineered: DBB highest, IPD lowest):")
print(_m.groupby("delivery_method")["overrun"].mean().sort_values(ascending=False).round(3))

StatementMeta(, 3b496799-df95-48ca-88fd-94359c3a5095, 16, Finished, Available, Finished, False)

Mean overrun by delivery method (engineered: DBB highest, IPD lowest):
delivery_method
Design-Bid-Build    1.315
CM Agency           1.257
CM at Risk          1.202
Design-Build        1.133
IPD                 1.100
Name: overrun, dtype: float64
